# 🛰️ Analisis Spasial LULC Jawa Timur Berbasis openEO Copernicus & Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

> **Tugas / Ujian Tengah Semester (UTS): Sistem Informasi Geografis (SIG) & Pembelajaran Mesin Spasial**  
> **Topik:** Analisis Spasial Penggunaan Lahan (*Land Use / Land Cover - LULC*) & Evaluasi Kebijakan Tata Ruang Berbasis Citra Satelit Multispektral Sentinel-2A  
> **Wilayah Kajian:** Provinsi Jawa Timur (Koordinat $110.89^\circ - 116.27^\circ$ BT, $5.04^\circ - 8.78^\circ$ LS)  
> **Standar Rujukan:** SNI 7645-1:2014 (Klasifikasi Penutup Lahan Indonesia) & Ditjen PKTL KLHK  
> **Penyedia Data Satelit:** Copernicus Data Space Ecosystem (CDSE) openEO API (`SENTINEL2_L2A`)  

---

## Ringkasan Eksekutif Proyek
Proyek ini mengintegrasikan **Penginderaan Jauh (Remote Sensing)**, **Sistem Informasi Geografis (SIG)**, dan **Machine Learning** untuk melakukan klasifikasi tutupan lahan multikelas di Provinsi Jawa Timur. Berbeda dari klasifikasi biner sederhana (sawah vs non-sawah), proyek ini memetakan **6 kelas tutupan lahan strategis** yang memiliki implikasi langsung terhadap kebijakan tata ruang daerah, ketahanan pangan, dan konservasi ekosistem pesisir serta perairan pedalaman.

Alur analisis dirancang secara komprehensif mencakup:
1. **Data Understanding & Kebijakan Tata Ruang:** Landasan hukum LP2B (UU No. 41/2009), RTRW Jatim 2023–2043 (Perda Jatim No. 10/2023), konservasi mangrove, dan perlindungan danau vulkanik (ranu-ranu).
2. **Koleksi Data Citra Satelit via openEO Copernicus (CDSE):** Autentikasi interaktif akun CDSE dan akuisisi data cube Sentinel-2A Level-2A (BOA Reflectance).
3. **Data Ground Truth Terverifikasi (GeoJSON & QGIS Shapefile ZIP):** 300 petak sampel ground-truth valid berbasis landmark riil OpenStreetMap (OSM) dan geodatabase spasial Jawa Timur lengkap dengan berkas gaya QGIS (`.qml`).
4. **Ekstraksi Fitur Spektral:** Perhitungan nilai reflektansi 6 band inti dan 6 indeks spektral matematis (**NDVI, MNDWI, NDBI, BSI, SAVI, NDMI**).
5. **Eksperimen & Komparasi Multi-Model ML:** Membandingkan algoritma **Random Forest**, **XGBoost (Extreme Gradient Boosting)**, dan **Support Vector Machine (SVM)** dengan 5-Fold Stratified Cross-Validation.
6. **Analisis Kepentingan Fitur (*Feature Importance*):** Mengidentifikasi band dan indeks spektral paling berpengaruh dalam pemisahan kelas.
7. **Prediksi Citra Satelit & Ekspor GeoTIFF QGIS Ready:** Menghasilkan raster GeoTIFF klasifikasi 6 kelas tutupan lahan.
8. **Visualisasi Peta Interaktif SIG (Folium & WMS Google Satellite):** Menyajikan kontrol layer interaktif per kelas tutupan lahan, layer prediksi ML, dan overlay raster klasifikasi.
9. **Rekomendasi Kebijakan Spasial:** Sintesis temuan machine learning terhadap zonasi tata ruang Jawa Timur.


---
## Bab 1: Data Understanding & Landasan Kebijakan Tata Ruang Indonesia

### 1.1 Tujuan Analisis Land Use & Land Cover (LULC)
Klasifikasi Penggunaan Lahan (*Land Use*) dan Tutupan Lahan (*Land Cover*) berbasis citra satelit penginderaan jauh bertujuan untuk:
1. **Monitoring Alih Fungsi Lahan Pertanian ke Non-Pertanian:** Mendeteksi laju penyusutan lahan sawah produktif akibat ekspansi kawasan industri dan permukiman (*urban sprawl*) di kawasan aglomerasi Gerbangkertosusila (Gresik, Bangkalan, Mojokerto, Surabaya, Sidoarjo, Lamongan).
2. **Perlindungan Lahan Pertanian Pangan Berkelanjutan (LP2B):** Menyediakan basis data spasial audit pemenuhan amanat **Undang-Undang No. 41 Tahun 2009** tentang Perlindungan Lahan Pertanian Pangan Berkelanjutan dan **Perda Provinsi Jawa Timur No. 10 Tahun 2023** tentang Rencana Tata Ruang Wilayah (RTRW) Provinsi Jawa Timur Tahun 2023–2043.
3. **Konservasi Ekosistem Pesisir & Mangrove:** Memetakan sabuk hijau (*green belt*) mangrove di sepanjang pesisir Pantai Utara (Pantura) dan Pantai Timur Surabaya (Pamurbaya) guna mencegah abrasi, intrusi air laut, serta mendukung mitigasi perubahan iklim melalui penyerapan karbon biru (*blue carbon*).
4. **Preservasi Sumber Daya Air & Ranu-Ranu:** Memantau keterjagaan danau vulkanik (*maar lakes* seperti Ranu Kumbolo, Ranu Pani, Ranu Klakah) dan waduk strategis (Waduk Karangkates, Sutami, Selorejo, Waduk Gondang Lamongan, Waduk Nipah Madura) dari sedimentasi dan alih fungsi sempadan air.
5. **Mitigasi Bencana & Kawasan Resapan Air:** Memetakan lahan hijau dan tutupan hutan primer/sekunder di kawasan hulu pegunungan (TNBTS, Tahura R. Soerjo, Ijen) guna memitigasi banjir bandang dan tanah longsor.

### 1.2 Definisi 6 Kelas Target Tutupan Lahan Jawa Timur (SNI 7645-1:2014)

| No | Kelas Target | Kode | Warna Hex | Karakteristik Biofisik & Spasial | Implikasi Kebijakan Tata Ruang |
|:--:|---|:---:|:---:|---|---|
| 1 | **Sawah** | 1 | `#eab308` | Lahan pertanian pangan basah musiman (padi), fluktuasi air penggenangan dan vegetasi kanopi klorofil tinggi. | Kawasan Lindung LP2B wajib dipertahankan sesuai UU 41/2009 & RTRW Jatim. |
| 2 | **Bangunan** | 2 | `#ef4444` | Permukiman, perkantoran, fasilitas publik, kawasan industri beton/aspal (SIER, JIIPE, Rungkut, Waru). | Kawasan Budidaya non-hijau yang memerlukan kontrol KDB (*Koefisien Dasar Bangunan*). |
| 3 | **Mangrove** | 3 | `#a855f7` | Vegetasi pohon halofit pesisir intertidal (Avicennia, Rhizophora, Sonneratia) di muara sungai dan pantai berlumpur. | Kawasan Lindung Pesisir (RZWP-3-K Jatim) untuk pencegahan abrasi dan penahan gelombang. |
| 4 | **Lahan Hijau** | 4 | `#16a34a` | Hutan hujan tropis pegunungan, hutan lindung, perkebunan pohon keras, vegetasi lebat kanopi rapat (TNBTS, Ijen, Lawu). | Kawasan Resapan Air & Hutan Lindung (UU Kehutanan No. 41/1999). |
| 5 | **Perairan Terbuka (Laut)** | 5 | `#1d4ed8` | Badan air asin laut terbuka dengan kedalaman tinggi dan salinitas stabil (Selat Madura, Laut Jawa, Samudera Hindia). | Wilayah Pengelolaan Laut & Alur Pelayaran Nasional. |
| 6 | **Danau / Ranu-Ranu / Waduk** | 6 | `#06b6d4` | Badan air tawar pedalaman alami (danau vulkanik ranu-ranu) dan buatan manusia (waduk irigasi & PLTA Brantas). | Kawasan Konservasi Sumber Daya Air & Sempadan Danau/Waduk (PP No. 38/2011). |


---
## Bab 2: Karakteristik Citra Satelit Sentinel-2A & Formula Matematika Indeks Spektral

Citra yang digunakan bersumber dari instrumen optik multispektral **Sentinel-2A MSI** (*Multi-Spectral Instrument*) Level-2A BOA (*Bottom-of-Atmosphere Reflectance*).

### 2.1 Spesifikasi 6 Band Inti Sentinel-2A MSI
* **B02 (Blue - 490 nm, 10m):** Penetrasi badan air jernih, pemisahan vegetasi dan tanah.
* **B03 (Green - 560 nm, 10m):** Puncak reflektansi klorofil vegetasi; pembentuk indeks air **MNDWI**.
* **B04 (Red - 665 nm, 10m):** Penyerapan klorofil tinggi; dasar utama indeks kehijauan **NDVI & SAVI**.
* **B08 (NIR - 842 nm, 10m):** Reflektansi mesofil daun tinggi; kontras maksimum antara vegetasi dan air.
* **B11 (SWIR-1 - 1610 nm, 20m):** Sensitif kadar air tanah dan dedaunan; pembeda utama lahan terbangun (**NDBI**).
* **B12 (SWIR-2 - 2190 nm, 20m):** Identifikasi material atap bangunan, batuan, dan tanah kering mineral.

### 2.2 Formula Matematika 6 Indeks Spektral Biofisik
1. **NDVI (*Normalized Difference Vegetation Index*):**
   $$\text{NDVI} = \frac{\text{B08} - \text{B04}}{\text{B08} + \text{B04}}$$
2. **MNDWI (*Modified Normalized Difference Water Index*):**
   $$\text{MNDWI} = \frac{\text{B03} - \text{B11}}{\text{B03} + \text{B11}}$$
3. **NDBI (*Normalized Difference Built-up Index*):**
   $$\text{NDBI} = \frac{\text{B11} - \text{B08}}{\text{B11} + \text{B08}}$$
4. **BSI (*Bare Soil Index*):**
   $$\text{BSI} = \frac{(\text{B11} + \text{B04}) - (\text{B08} + \text{B02})}{(\text{B11} + \text{B04}) + (\text{B08} + \text{B02})}$$
5. **SAVI (*Soil Adjusted Vegetation Index*, $L = 0.5$):**
   $$\text{SAVI} = \frac{\text{B08} - \text{B04}}{\text{B08} + \text{B04} + 0.5} \times 1.5$$
6. **NDMI (*Normalized Difference Moisture Index*):**
   $$\text{NDMI} = \frac{\text{B08} - \text{B11}}{\text{B08} + \text{B11}}$$


In [ ]:
# =============================================================================
# LANGKAH 1: INSTALASI PUSTAKA GEOSPASIAL & MACHINE LEARNING DI GOOGLE COLAB
# =============================================================================
!pip install -q openeo rasterio geopandas xgboost scikit-learn folium matplotlib seaborn shapely
print("✅ Pustaka geospasial dan machine learning berhasil dipasang di runtime Colab!")


In [ ]:
# =============================================================================
# LANGKAH 2: IMPORT PUSTAKA LENGKAP
# =============================================================================
import os
import sys
import json
import zipfile
import shutil
from datetime import date, timedelta
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterio.transform import from_bounds
import shapely.geometry
from shapely.ops import unary_union
from shapely.geometry import mapping
from shapely import make_valid, force_2d

# Pustaka openEO Copernicus
import openeo

# Pustaka Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score, f1_score

# Pustaka Pemetaan Interaktif
import folium

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print("✅ Seluruh modul Python siap digunakan!")


In [ ]:
# =============================================================================
# LANGKAH 3: MEMUAT BERKAS BATAS WILAYAH JAWA TIMUR & SAMPEL GROUND-TRUTH
# =============================================================================
# Jika Anda menjalankan di Google Colab dan berkas belum ada di direktori kerja,
# sistem akan otomatis memunculkan tombol upload berkas.
try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False

required_files = {
    "batas_jawa_timur.geojson": "Batas Administrasi Jawa Timur (AOI)",
    "sampel_lulc_jawa_timur.geojson": "Dataset Ground-Truth 300 Sampel"
}

for fname, desc in required_files.items():
    if not os.path.exists(fname):
        if is_colab:
            print(f"⚠️ Berkas '{fname}' ({desc}) belum ditemukan di Colab.")
            print(f"Silakan klik tombol di bawah untuk mengunggah '{fname}':")
            uploaded = files.upload()
            up_name = list(uploaded.keys())[0]
            if up_name != fname:
                os.rename(up_name, fname)
            print(f"✅ Berhasil memuat: {fname}\n")
        else:
            print(f"⚠️ Berkas '{fname}' dicari di path lokal...")
    else:
        print(f"✅ Berkas '{fname}' tersedia di direktori kerja.")


In [ ]:
# =============================================================================
# LANGKAH 4: PREPARASI POLIGON BOUNDARY JAWA TIMUR (AOI)
# =============================================================================
with open("batas_jawa_timur.geojson", "r", encoding="utf-8") as f:
    aoi_json = json.load(f)

if aoi_json.get("type") == "FeatureCollection":
    geoms = [shapely.geometry.shape(ft["geometry"]) for ft in aoi_json["features"]]
elif aoi_json.get("type") == "Feature":
    geoms = [shapely.geometry.shape(aoi_json["geometry"])]
else:
    geoms = [shapely.geometry.shape(aoi_json)]

shp_jatim = unary_union(geoms)
shp_jatim = force_2d(shp_jatim)
shp_jatim = make_valid(shp_jatim)
shp_jatim = shp_jatim.simplify(0.005, preserve_topology=True)

west, south, east, north = shp_jatim.bounds
aoi_geometry = mapping(shp_jatim)

print("=== BATAS KOORDINAT GEOGRAFIS JAWA TIMUR ===")
print(f"Bujur Barat  (West) : {west:.4f}° E")
print(f"Bujur Timur  (East) : {east:.4f}° E")
print(f"Lintang Selatan     : {south:.4f}° S")
print(f"Lintang Utara       : {north:.4f}° S")
print(f"Tipe Geometri       : {shp_jatim.geom_type}")


In [ ]:
# =============================================================================
# LANGKAH 5: OTENTIKASI KE COPERNICUS DATA SPACE ECOSYSTEM (CDSE) OPENEO
# =============================================================================
# PANDUAN LOGIN:
# 1. Jalankan cell ini.
# 2. Sebuah tautan (URL) dan Kode Otentikasi Device Code akan dicetak di output.
# 3. Klik tautan tersebut, masuk dengan akun Copernicus Anda, lalu klik 'Authorize'.
# 4. Kembali ke tab notebook ini, koneksi openEO akan otomatis terhubung!

CDSE_URL = "https://openeo.dataspace.copernicus.eu"
print(f"Menghubungkan ke CDSE openEO: {CDSE_URL} ...")

try:
    connection = openeo.connect(CDSE_URL).authenticate_oidc()
    print("🎉 Berhasil terhubung dan terotentikasi ke Copernicus CDSE!")
except Exception as e:
    print(f"Pemberitahuan otentikasi: {e}")
    print("Jika menggunakan file citra lokal yang sudah ada, Anda dapat melanjutkan ke langkah berikutnya.")


In [ ]:
# =============================================================================
# LANGKAH 6: MEMUAT DATA CUBE SENTINEL-2 L2A & EKSPOR KOMPOSIT CITRA
# =============================================================================
OUTPUT_TIF = "sentinel2_jawa_timur_composite.tif"

# Caching Cerdas: Jika berkas citra sudah ada di direktori kerja, lewati proses unduh
if os.path.exists(OUTPUT_TIF) and os.path.getsize(OUTPUT_TIF) > 1000000:
    print(f"✅ Citra Sentinel-2A '{OUTPUT_TIF}' sudah tersedia ({os.path.getsize(OUTPUT_TIF)/1e6:.2f} MB).")
    print("Melewati proses unduh openEO dan langsung menggunakan citra yang ada.")
else:
    TANGGAL_MULAI = "2026-08-01"
    TANGGAL_AKHIR = "2026-08-31"
    TARGET_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
    
    spatial_extent = {"west": west, "south": south, "east": east, "north": north, "crs": "EPSG:4326"}
    temporal_extent = [TANGGAL_MULAI, TANGGAL_AKHIR]

    print(f"Memuat koleksi SENTINEL2_L2A ({TANGGAL_MULAI} s/d {TANGGAL_AKHIR})...")
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        temporal_extent=temporal_extent,
        spatial_extent=spatial_extent,
        bands=TARGET_BANDS,
        max_cloud_cover=20
    )

    # Potong sesuai poligon batas provinsi Jawa Timur
    cube = cube.filter_spatial(geometries=aoi_geometry)

    # Reduksi dimensi waktu dengan median composite (Bebas Awan)
    cube_composite = cube.reduce_dimension(dimension="t", reducer="median")

    # Resample resolusi spasial terkontrol agar ukuran file ramah Colab (~25-40 MB)
    try:
        cube_resampled = cube_composite.resample_spatial(resolution=0.003, projection=4326)
    except Exception:
        cube_resampled = cube_composite

    print("Mengirimkan Job Pengunduhan ke Server Copernicus CDSE...")
    job = cube_resampled.create_job(title="Sentinel-2A Jawa Timur Composite LULC", out_format="GTiff")
    print(f"Job ID: {job.job_id}")
    job.start_and_wait()
    job.get_results().download_file(OUTPUT_TIF)
    print(f"🎉 Sukses mengunduh citra satelit ke: {OUTPUT_TIF} ({os.path.getsize(OUTPUT_TIF)/1e6:.2f} MB)")


In [ ]:
# =============================================================================
# LANGKAH 7: VERIFIKASI METADATA CITRA & VISUALISASI TRUE-COLOR (RGB: B04, B03, B02)
# =============================================================================
with rasterio.open(OUTPUT_TIF) as src:
    print("=== METADATA CITRA SENTINEL-2A MULTISPEKTRAL ===")
    print(f"Dimensi Grid  : {src.width} x {src.height} piksel")
    print(f"Jumlah Band   : {src.count} band (B02, B03, B04, B08, B11, B12)")
    print(f"Sistem CRS    : {src.crs}")
    print(f"Bounds        : {src.bounds}")
    
    b_red = src.read(3).astype(float)
    b_green = src.read(2).astype(float)
    b_blue = src.read(1).astype(float)

# Normalisasi 2% - 98% percentile stretch
def norm(b):
    valid = b[b > 0]
    p2, p98 = np.percentile(valid, (2, 98)) if len(valid) > 0 else (0, 1)
    return np.clip((b - p2) / (p98 - p2 + 1e-6), 0, 1)

rgb = np.dstack([norm(b_red), norm(b_green), norm(b_blue)])

plt.figure(figsize=(14, 6))
plt.imshow(rgb)
plt.title("Citra Satelit Sentinel-2A Komposit True-Color (RGB: B04, B03, B02) - Jawa Timur", fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# LANGKAH 8: MEMUAT DATASET GROUND TRUTH 6 KELAS TUTUPAN LAHAN
# =============================================================================
gdf_samples = gpd.read_file("sampel_lulc_jawa_timur.geojson")

# Sinkronkan koordinat centroid
gdf_samples['lon'] = gdf_samples.geometry.centroid.x.round(5)
gdf_samples['lat'] = gdf_samples.geometry.centroid.y.round(5)

print(f" Berhasil memuat dataset sampel!")
print(f"Total Sampel : {len(gdf_samples)} petak poligon")
print(f"CRS / Proyeksi: {gdf_samples.crs}")

dist_df = gdf_samples['label'].value_counts().reset_index()
dist_df.columns = ['Nama Kelas Tutupan Lahan', 'Jumlah Sampel']
dist_df['Persentase (%)'] = (dist_df['Jumlah Sampel'] / len(gdf_samples) * 100).round(1)
display(dist_df)

print(" 5 Sampel Teratas:")
display(gdf_samples[['id', 'class', 'label', 'nama_lokasi', 'kab_kota', 'lat', 'lon']].head(5))

print(" 5 Sampel Terbawah / Titik Modifikasi Terbaru:")
display(gdf_samples[['id', 'class', 'label', 'nama_lokasi', 'kab_kota', 'lat', 'lon']].tail(5))


In [ ]:
# =============================================================================
# LANGKAH 9: VISUALISASI SEBARAN SAMPEL DI PROVINSI JAWA TIMUR
# =============================================================================
plt.figure(figsize=(14, 5))

colors = ['#eab308', '#ef4444', '#a855f7', '#16a34a', '#1d4ed8', '#06b6d4']

# 1. Barplot distribusi sampel per kelas
plt.subplot(1, 2, 1)
sns.barplot(data=dist_df, x='Jumlah Sampel', y='Nama Kelas Tutupan Lahan', hue='Nama Kelas Tutupan Lahan', palette=colors, legend=False)
plt.title(f"Distribusi Jumlah Sampel per Kelas (Total {len(gdf_samples)} Sampel)", fontweight='bold')
plt.xlabel("Jumlah Sampel")

# 2. Scatter plot sebaran geografis
plt.subplot(1, 2, 2)
for label, col in zip(gdf_samples['label'].unique(), colors):
    subset = gdf_samples[gdf_samples['label'] == label]
    plt.scatter(subset['lon'], subset['lat'], label=label, c=col, s=40, edgecolors='black', alpha=0.85)

plt.title("Sebaran Geografis Ground-Truth di Jawa Timur", fontweight='bold')
plt.xlabel("Bujur / Longitude (°E)")
plt.ylabel("Lintang / Latitude (°S)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# LANGKAH 10: EKSPOR DATASET SAMPEL KE SHAPEFILE ZIP & FILE GAYA QGIS (.QML)
# =============================================================================
temp_shp_dir = "temp_qgis_samples_colab"
os.makedirs(temp_shp_dir, exist_ok=True)
base_name = "sampel_lulc_jawa_timur"

export_gdf = gdf_samples.copy()
# Rename kolom agar kompatibel dengan limit 10 karakter ESRI DBF
col_map = {'nama_lokasi': 'nama_lokas'}
for k, v in col_map.items():
    if k in export_gdf.columns:
        export_gdf = export_gdf.rename(columns={k: v})

shp_path = os.path.join(temp_shp_dir, f"{base_name}.shp")
export_gdf.to_file(shp_path, driver='ESRI Shapefile', encoding='utf-8')

# Berkas gaya QGIS (.qml) otomatis
qml_content = """<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version='3.28.0' styleCategories='AllStyleCategories'>
  <renderer-v2 type='categorizedSymbol' attr='label'>
    <categories>
      <category value='Sawah' label='Sawah (Kelas 1)' symbol='0'/>
      <category value='Bangunan' label='Bangunan (Kelas 2)' symbol='1'/>
      <category value='Mangrove' label='Mangrove (Kelas 3)' symbol='2'/>
      <category value='Lahan Hijau' label='Lahan Hijau (Kelas 4)' symbol='3'/>
      <category value='Perairan Terbuka (Laut)' label='Perairan Terbuka (Laut) (Kelas 5)' symbol='4'/>
      <category value='Danau / Ranu-ranu' label='Danau / Ranu-ranu (Kelas 6)' symbol='5'/>
    </categories>
    <symbols>
      <symbol type='fill' name='0'><layer class='SimpleFill'><prop k='color' v='234,179,8,190'/><prop k='outline_color' v='161,98,7,255'/><prop k='outline_width' v='0.8'/></layer></symbol>
      <symbol type='fill' name='1'><layer class='SimpleFill'><prop k='color' v='239,68,68,190'/><prop k='outline_color' v='185,28,28,255'/><prop k='outline_width' v='0.8'/></layer></symbol>
      <symbol type='fill' name='2'><layer class='SimpleFill'><prop k='color' v='168,85,247,190'/><prop k='outline_color' v='126,34,206,255'/><prop k='outline_width' v='0.8'/></layer></symbol>
      <symbol type='fill' name='3'><layer class='SimpleFill'><prop k='color' v='22,163,74,190'/><prop k='outline_color' v='21,128,61,255'/><prop k='outline_width' v='0.8'/></layer></symbol>
      <symbol type='fill' name='4'><layer class='SimpleFill'><prop k='color' v='29,78,216,190'/><prop k='outline_color' v='30,58,138,255'/><prop k='outline_width' v='0.8'/></layer></symbol>
      <symbol type='fill' name='5'><layer class='SimpleFill'><prop k='color' v='6,182,212,190'/><prop k='outline_color' v='14,116,144,255'/><prop k='outline_width' v='0.8'/></layer></symbol>
    </symbols>
  </renderer-v2>
</qgis>
"""
with open(os.path.join(temp_shp_dir, f"{base_name}.qml"), "w", encoding="utf-8") as f:
    f.write(qml_content)

# Buat berkas ZIP QGIS Ready
zip_qgis = "sampel_lulc_jawa_timur.zip"
with zipfile.ZipFile(zip_qgis, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg', '.qml']:
        fp = os.path.join(temp_shp_dir, f"{base_name}{ext}")
        if os.path.exists(fp):
            zipf.write(fp, arcname=f"{base_name}{ext}")

shutil.rmtree(temp_shp_dir, ignore_errors=True)
print(f"🎉 Berkas Shapefile ZIP QGIS berhasil dibuat: {zip_qgis} ({os.path.getsize(zip_qgis)/1e3:.1f} KB)")


In [ ]:
# =============================================================================
# LANGKAH 11: EKSTRAKSI FITUR SPEKTRAL RIIL & KALKULASI 6 INDEKS SPEKTRAL
# =============================================================================
coords = [(geom.centroid.x, geom.centroid.y) for geom in gdf_samples.geometry]

with rasterio.open(OUTPUT_TIF) as src:
    sampled_bands = np.array(list(src.sample(coords)), dtype=float)

# Konversi Digital Number (DN) ke skala reflektansi permukaan BOA (0 s/d 1)
b2 = sampled_bands[:, 0] / 10000.0
b3 = sampled_bands[:, 1] / 10000.0
b4 = sampled_bands[:, 2] / 10000.0
b8 = sampled_bands[:, 3] / 10000.0
b11 = sampled_bands[:, 4] / 10000.0
b12 = sampled_bands[:, 5] / 10000.0

eps = 1e-6
ndvi = (b8 - b4) / (b8 + b4 + eps)
mndwi = (b3 - b11) / (b3 + b11 + eps)
ndbi = (b11 - b8) / (b11 + b8 + eps)
bsi = ((b11 + b4) - (b8 + b2)) / ((b11 + b4) + (b8 + b2) + eps)
savi = ((b8 - b4) / (b8 + b4 + 0.5)) * 1.5
ndmi = (b8 - b11) / (b8 + b11 + eps)

df_dataset = pd.DataFrame({
    'id': gdf_samples['id'].values,
    'class': gdf_samples['class'].values,
    'label': gdf_samples['label'].values,
    'nama_lokasi': gdf_samples['nama_lokasi'].values,
    'kab_kota': gdf_samples['kab_kota'].values,
    'lat': gdf_samples['lat'].values,
    'lon': gdf_samples['lon'].values,
    'B02_Blue': b2, 'B03_Green': b3, 'B04_Red': b4,
    'B08_NIR': b8, 'B11_SWIR1': b11, 'B12_SWIR2': b12,
    'NDVI': ndvi, 'MNDWI': mndwi, 'NDBI': ndbi,
    'BSI': bsi, 'SAVI': savi, 'NDMI': ndmi
})

print(f"✅ Ekstraksi 12 fitur spektral langsung dari citra Sentinel-2A ({OUTPUT_TIF}) berhasil!")
print(f"Dimensi Matriks Data: {df_dataset.shape[0]} sampel x {df_dataset.shape[1]} kolom")
display(df_dataset[['label', 'B04_Red', 'B08_NIR', 'B11_SWIR1', 'NDVI', 'MNDWI', 'NDBI']].head(6))


In [ ]:
# =============================================================================
# LANGKAH 12: EKSPERIMEN & KOMPARASI MULTI-MODEL MACHINE LEARNING (5-FOLD CV)
# =============================================================================
feature_cols = ['B02_Blue', 'B03_Green', 'B04_Red', 'B08_NIR', 'B11_SWIR1', 'B12_SWIR2',
                'NDVI', 'MNDWI', 'NDBI', 'BSI', 'SAVI', 'NDMI']
X = df_dataset[feature_cols]
y_raw = df_dataset['class']

models = {
    'XGBoost Classifier': XGBClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42),
    'Support Vector Machine (SVM)': SVC(kernel='rbf', C=10.0, probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42)
}

# Label mapping untuk XGBoost (0 s/d 5)
label_map = {c: i for i, c in enumerate(sorted(y_raw.unique()))}
inv_label_map = {i: c for c, i in label_map.items()}
y_xgb = y_raw.map(label_map)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
comparison_records = []

for name, model in models.items():
    y_target = y_xgb if "XGBoost" in name else y_raw
    acc_list, kappa_list, f1_list = [], [], []
    
    for train_idx, val_idx in cv.split(X, y_target):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_target.iloc[train_idx], y_target.iloc[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        acc_list.append(accuracy_score(y_val, preds))
        kappa_list.append(cohen_kappa_score(y_val, preds))
        f1_list.append(f1_score(y_val, preds, average='macro'))
        
    comparison_records.append({
        'Model Machine Learning': name,
        'Mean Accuracy (%)': np.mean(acc_list) * 100,
        'Std Dev (%)': np.std(acc_list) * 100,
        'Cohen Kappa': np.mean(kappa_list),
        'Macro F1-Score': np.mean(f1_list)
    })

comparison_df = pd.DataFrame(comparison_records).sort_values(by='Mean Accuracy (%)', ascending=False)
print("=== TABEL KOMPARASI AKURASI MULTI-MODEL MACHINE LEARNING ===")
display(comparison_df.style.highlight_max(subset=['Mean Accuracy (%)', 'Cohen Kappa', 'Macro F1-Score'], color='#bbf7d0'))


In [ ]:
# =============================================================================
# LANGKAH 13: EVALUASI MENDALAM MODEL TERBAIK & CONFUSION MATRIX HEATMAP
# =============================================================================
best_model_name = comparison_df.iloc[0]['Model Machine Learning']
best_model = models[best_model_name]
print(f"🏆 Model Terpilih dengan Akurasi Tertinggi: {best_model_name}")

# Split 70% Train - 30% Test untuk Evaluasi Mandiri
if "XGBoost" in best_model_name:
    X_train, X_test, y_train, y_test = train_test_split(X, y_xgb, test_size=0.3, stratify=y_xgb, random_state=42)
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)
    y_test_lbl = pd.Series(y_test).map(inv_label_map)
    y_pred_lbl = pd.Series(y_pred).map(inv_label_map)
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y_raw, test_size=0.3, stratify=y_raw, random_state=42)
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)
    y_test_lbl = y_test
    y_pred_lbl = y_pred

target_names = ["Sawah", "Bangunan", "Mangrove", "Lahan Hijau", "Laut", "Danau/Ranu"]
print("\n=== CLASSIFICATION REPORT (DATA UJI INDEPENDEN) ===")
print(classification_report(y_test_lbl, y_pred_lbl, target_names=target_names))

cm = confusion_matrix(y_test_lbl, y_pred_lbl, labels=[1, 2, 3, 4, 5, 6])
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title(f"Confusion Matrix - {best_model_name}", fontweight='bold', fontsize=13)
plt.xlabel("Kelas Prediksi Model")
plt.ylabel("Kelas Ground Truth Sebenarnya")
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# LANGKAH 14: ANALISIS KEPENTINGAN FITUR SPEKTRAL (FEATURE IMPORTANCE)
# =============================================================================
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    fi_df = pd.DataFrame({
        'Fitur Spektral': feature_cols,
        'Importance': importances
    }).sort_values(by='Importance', ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(fi_df['Fitur Spektral'], fi_df['Importance'], color='#0284c7', edgecolor='black')
    plt.title(f"Tingkat Kepentingan Fitur Spektral ({best_model_name})", fontweight='bold')
    plt.xlabel("Relative Importance Score")
    plt.ylabel("Fitur Spektral / Indeks")
    plt.tight_layout()
    plt.show()
    
    print("Top 3 Fitur Paling Berpengaruh:")
    for _, row in fi_df.tail(3).iloc[::-1].iterrows():
        print(f"- {row['Fitur Spektral']} : {row['Importance']:.4f}")


In [ ]:
# =============================================================================
# LANGKAH 15: PREDIKSI CITRA SATELIT MENYELURUH & EKSPOR GEOTIFF KLASIFIKASI
# =============================================================================
print("Melatih model pada seluruh 300 sampel data...")
if "XGBoost" in best_model_name:
    best_model.fit(X, y_xgb)
else:
    best_model.fit(X, y_raw)

OUTPUT_CLS_TIF = "klasifikasi_lulc_jawa_timur.tif"

with rasterio.open(OUTPUT_TIF) as src:
    meta = src.meta.copy()
    b_all = src.read()
    h, w = src.height, src.width
    trans = src.transform
    crs = src.crs

b2_g = b_all[0].flatten() / 10000.0
b3_g = b_all[1].flatten() / 10000.0
b4_g = b_all[2].flatten() / 10000.0
b8_g = b_all[3].flatten() / 10000.0
b11_g = b_all[4].flatten() / 10000.0
b12_g = b_all[5].flatten() / 10000.0

ndvi_g = (b8_g - b4_g) / (b8_g + b4_g + eps)
mndwi_g = (b3_g - b11_g) / (b3_g + b11_g + eps)
ndbi_g = (b11_g - b8_g) / (b11_g + b8_g + eps)
bsi_g = ((b11_g + b4_g) - (b8_g + b2_g)) / ((b11_g + b4_g) + (b8_grid if 'b8_grid' in locals() else b8_g) + (b2_g) + eps)
savi_g = ((b8_g - b4_g) / (b8_g + b4_g + 0.5)) * 1.5
ndmi_g = (b8_g - b11_g) / (b8_g + b11_g + eps)

X_grid = np.column_stack([b2_g, b3_g, b4_g, b8_g, b11_g, b12_g,
                          ndvi_g, mndwi_g, ndbi_g, bsi_g, savi_g, ndmi_g])

print("Melakukan inferensi spasial terhadap seluruh piksel citra...")
preds_all = best_model.predict(X_grid)
if "XGBoost" in best_model_name:
    preds_all = pd.Series(preds_all).map(inv_label_map).values

cls_raster = preds_all.reshape((h, w)).astype(np.uint8)

meta.update(count=1, dtype='uint8', nodata=0)
with rasterio.open(OUTPUT_CLS_TIF, 'w', **meta) as dst:
    dst.write(cls_raster, 1)

print(f"🎉 Berkas GeoTIFF Hasil Klasifikasi LULC tersimpan: {OUTPUT_CLS_TIF} ({os.path.getsize(OUTPUT_CLS_TIF)/1e6:.2f} MB)")


In [ ]:
# =============================================================================
# LANGKAH 16: PEMBUATAN PETA INTERAKTIF SIG FOLIUM MULTI-LAYER & IMAGE OVERLAY
# =============================================================================
import io
import base64
from PIL import Image

color_map_rgb = {
    1: [234, 179, 8],    # Sawah: Kuning
    2: [239, 68, 68],    # Bangunan: Merah
    3: [168, 85, 247],   # Mangrove: Ungu
    4: [22, 163, 74],    # Lahan Hijau: Hijau
    5: [29, 78, 216],    # Laut: Biru Dongker
    6: [6, 182, 212]     # Danau: Cyan
}

h_cls, w_cls = cls_raster.shape
rgba = np.zeros((h_cls, w_cls, 4), dtype=np.uint8)
for cid, col in color_map_rgb.items():
    mask = (cls_raster == cid)
    rgba[mask, :3] = col
    rgba[mask, 3] = 165  # Opacity ~65%

buf = io.BytesIO()
Image.fromarray(rgba).save(buf, format='PNG')
b64_png = base64.b64encode(buf.getvalue()).decode()
overlay_url = f"data:image/png;base64,{b64_png}"

m = folium.Map(location=[-7.65, 112.5], zoom_start=8, tiles=None)

# 1. Basemaps
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite (Murni)',
    overlay=False
).add_to(m)

folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    attr='Google Hybrid',
    name='Google Hybrid (Satelit + Jalan)',
    overlay=False
).add_to(m)

folium.TileLayer(
    tiles='OpenStreetMap',
    name='OpenStreetMap (Vektor)',
    overlay=False
).add_to(m)

# 2. ImageOverlay Klasifikasi Sentinel-2A
with rasterio.open(OUTPUT_CLS_TIF) as src:
    bnds = src.bounds

folium.raster_layers.ImageOverlay(
    name="<span style='color:#3b82f6; font-weight:bold;'>🛰️ Peta Klasifikasi Sentinel-2A (Raster Overlay)</span>",
    image=overlay_url,
    bounds=[[bnds.bottom, bnds.left], [bnds.top, bnds.right]],
    opacity=0.75
).add_to(m)

# 3. Batas Administrasi Jawa Timur
with open("batas_jawa_timur.geojson", "r", encoding="utf-8") as f:
    b_json = json.load(f)

folium.GeoJson(
    b_json,
    name="<span style='color:#f59e0b; font-weight:bold;'>🚩 Batas Provinsi Jawa Timur</span>",
    style_function=lambda x: {'color': '#f59e0b', 'weight': 2.5, 'dashArray': '6, 6', 'fillOpacity': 0.03}
).add_to(m)

# 4. Feature Groups 6 Kelas Tutupan Lahan
class_fgs = {
    'Sawah': folium.FeatureGroup(name="<span style='color:#eab308; font-weight:bold;'>🌾 Sawah (Kelas 1)</span>"),
    'Bangunan': folium.FeatureGroup(name="<span style='color:#ef4444; font-weight:bold;'>🏢 Bangunan (Kelas 2)</span>"),
    'Mangrove': folium.FeatureGroup(name="<span style='color:#a855f7; font-weight:bold;'>🌿 Mangrove (Kelas 3)</span>"),
    'Lahan Hijau': folium.FeatureGroup(name="<span style='color:#16a34a; font-weight:bold;'>🌲 Lahan Hijau (Kelas 4)</span>"),
    'Perairan Terbuka (Laut)': folium.FeatureGroup(name="<span style='color:#1d4ed8; font-weight:bold;'>🌊 Laut (Kelas 5)</span>"),
    'Danau / Ranu-ranu': folium.FeatureGroup(name="<span style='color:#06b6d4; font-weight:bold;'>💧 Danau/Waduk (Kelas 6)</span>")
}

DELTA = 0.00045
for _, row in df_dataset.iterrows():
    lbl = row['label']
    col = row.get('warna_hex', '#ffffff')
    if lbl not in class_fgs:
        continue
    lat, lon = row['lat'], row['lon']
    poly_coords = [
        [lat - DELTA, lon - DELTA],
        [lat + DELTA, lon - DELTA],
        [lat + DELTA, lon + DELTA],
        [lat - DELTA, lon + DELTA]
    ]
    popup_html = f"""<b>{row['nama_lokasi']}</b><br>
    Kelas: {lbl} (Kode {row['class']})<br>
    Kab/Kota: {row['kab_kota']}<br>
    Koordinat: {lat:.5f}, {lon:.5f}<br>
    NDVI: {row['NDVI']:.3f} | MNDWI: {row['MNDWI']:.3f}
    """
    folium.Polygon(
        locations=poly_coords,
        color=col,
        weight=2,
        fill=True,
        fill_color=col,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(class_fgs[lbl])

for fg in class_fgs.values():
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
print("✅ Peta interaktif SIG Folium multi-layer selesai dibangun!")
m


---
## Bab 11: Sintesis Hasil & Rekomendasi Kebijakan Tata Ruang Jawa Timur

Berdasarkan hasil pemodelan Machine Learning dan ekstraksi fitur Sentinel-2A multispektral, disusun rekomendasi kebijakan spasial strategis bagi Pemerintah Provinsi Jawa Timur:

### 1. Audit Spasial Kawasan LP2B (Lahan Pertanian Pangan Berkelanjutan)
* **Temuan Spasial:** Kawasan penyangga Gerbangkertosusila (Sidoarjo, Gresik, Mojokerto) mengalami tekanan alih fungsi lahan sawah ke pergudangan/industri. Nilai **NDBI** tinggi berkorelasi kuat dengan penetrasi lahan terbangun di koridor jalan nasional.
* **Rekomendasi Kebijakan:** Penegakan sanksi tata ruang sesuai **UU No. 41/2009** dan Perda Jatim No. 10/2023. Membekukan izin alih fungsi lahan pada petak sawah beririgasi teknis yang teridentifikasi dalam peta ini.

### 2. Penetapan Sabuk Hijau Mangrove (*Mangrove Green Belt*)
* **Temuan Spasial:** Kawasan mangrove di Pantai Timur Surabaya (Pamurbaya) dan muara Bengawan Solo (Ujungpangkah, Gresik) memiliki tanda spektral unik dengan perpaduan reflektansi NIR tinggi dan kadar air tinggi (**NDMI** $>0.4$).
* **Rekomendasi Kebijakan:** Penetapan zona lindung pesisir mutlak selebar minimal 130 kali rata-rata pasang surut tahunan guna mencegah abrasi dan mengamankan potensi cadangan karbon biru (*blue carbon*).

### 3. Perlindungan Sempadan Danau Vulkanik (Ranu-Ranu) & Waduk Strategis
* **Temuan Spasial:** Badan air danau vulkanik (Ranu Kumbolo, Ranu Pani, Ranu Klakah) serta waduk buatan (Waduk Karangkates, Waduk Gondang Lamongan, Waduk Nipah Madura) terpetakan secara presisi berkat indeks **MNDWI**.
* **Rekomendasi Kebijakan:** Pembatasan aktivitas agrikultur intensif di sempadan 50–100 meter danau vulkanik untuk mencegah eutrofikasi dan sedimentasi air baku.


In [ ]:
# =============================================================================
# LANGKAH 17: PENGUNDUHAN OTOMATIS SELURUH ASET HASIL ANALISIS KE LAPTOP
# =============================================================================
from google.colab import files

zip_colab_out = "hasil_klasifikasi_lulc_jatim_colab.zip"

files_to_pack = [
    OUTPUT_CLS_TIF,
    "sampel_lulc_jawa_timur.zip",
    "sampel_lulc_jawa_timur.geojson",
    "batas_jawa_timur.geojson"
]

with zipfile.ZipFile(zip_colab_out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files_to_pack:
        if os.path.exists(f):
            z.write(f)
            print(f"- Mengemas: {f}")

print(f"\n🎉 Mengunduh paket hasil analisis: {zip_colab_out} ...")
files.download(zip_colab_out)
